In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [2]:
# Using sample data for testing purposes
data_path = Path("../../data/sample_resources.csv")
agent_output_path = Path("output/recommendations.json")
output_dir = Path("output")

print(f"Data path: {data_path}")
print(f"Agent output path: {agent_output_path}")
print(f"Output directory: {output_dir}")


Data path: ..\..\data\sample_resources.csv
Agent output path: output\recommendations.json
Output directory: output


In [3]:
df = pd.read_csv(data_path)
print(f"Rows: {len(df)}")
display(df.head())

Rows: 6


,resource_id,team,environment,service,monthly_cost,cpu_utilization,memory_utilization,uptime_required,policy_locked,schedule_eligible,storage_tier_eligible
0,vm-prod-01,core-platform,prod,vm,420.0,18,31,24x7,False,False,False
1,vm-prod-02,payments,prod,vm,510.0,44,62,24x7,True,False,False
2,vm-dev-01,analytics,dev,vm,260.0,12,28,business-hours,False,True,False
3,vm-test-01,marketing,test,vm,180.0,16,24,business-hours,False,True,False
4,storage-01,data-eng,prod,storage,190.0,0,0,24x7,False,False,True


In [4]:
def to_bool(series):
    if series.dtype == bool:
        return series
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
        .fillna(False)
    )

In [5]:
df["monthly_cost"] = pd.to_numeric(df["monthly_cost"], errors="coerce")
df["cpu_utilization"] = pd.to_numeric(df["cpu_utilization"], errors="coerce")
df["memory_utilization"] = pd.to_numeric(df["memory_utilization"], errors="coerce")
df["policy_locked"] = to_bool(df["policy_locked"])
df["schedule_eligible"] = to_bool(df["schedule_eligible"])
df["storage_tier_eligible"] = to_bool(df["storage_tier_eligible"])

In [6]:
quality_checks = {
    "null_resource_id": int(df["resource_id"].isna().sum()),
    "null_monthly_cost": int(df["monthly_cost"].isna().sum()),
    "negative_monthly_cost": int((df["monthly_cost"] < 0).sum()),
    "duplicate_resource_id": int(df["resource_id"].duplicated().sum()),
}
print("\nData quality checks")
display(pd.DataFrame([quality_checks]))
if any(quality_checks.values()):
    print("Warning: data quality issues detected. Review before trusting baseline outputs.")
else:
    print("Data quality checks passed.")


Data quality checks


,null_resource_id,null_monthly_cost,negative_monthly_cost,duplicate_resource_id
0,0,0,0,0


Data quality checks passed.
